In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense




In [2]:

# Load dataset
data = pd.read_csv("Housing.csv")

print(data.head())
print(data.shape)
print(data.isnull().sum())




      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  
(545, 13)
price               0
area                0
bedrooms            0
bathrooms           0
s

In [3]:
# Remove missing values
data = data.dropna()


# Convert yes/no into 1/0
yes_no_columns = [
    "mainroad",
    "guestroom",
    "basement",
    "hotwaterheating",
    "airconditioning",
    "prefarea"
]

for column in yes_no_columns:
    data[column] = data[column].map({
        "no": 0,
        "yes": 1
    })


# Convert furnishing status
data["furnishingstatus"] = data["furnishingstatus"].map({
    "unfurnished": 0,
    "semi-furnished": 1,
    "furnished": 2
})


# Inputs and output
X = data.drop(columns=["price"])
y = data[["price"]]


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# Scale inputs and output
X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)


In [8]:
X_train_scaled.shape[1]

12

In [10]:


# Create ANN
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(10, activation="relu"),
    Dense(5, activation="relu"),
    Dense(1)
])

In [ ]:



# Compile model
model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)


# Train model
history = model.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=100,
    batch_size=32,
    validation_split=0.20,
    verbose=1
)


# Make predictions
predictions_scaled = model.predict(
    X_test_scaled,
    verbose=0
)

predictions = y_scaler.inverse_transform(
    predictions_scaled
)


# Evaluation
mae = mean_absolute_error(
    y_test,
    predictions
)

r2 = r2_score(
    y_test,
    predictions
)

print(f"Mean Absolute Error: {mae:.2f}")
print(f"R2 Score: {r2:.2f}")


# Compare actual and predicted prices
results = pd.DataFrame({
    "Actual Price": y_test["price"].values,
    "Predicted Price": predictions.flatten()
})



Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - loss: 1.0984 - val_loss: 1.0843
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.9465 - val_loss: 0.9819
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.8429 - val_loss: 0.9011
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.7678 - val_loss: 0.8307
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.6943 - val_loss: 0.7820
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.6347 - val_loss: 0.7436
Epoch 7/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.5917 - val_loss: 0.7097
Epoch 8/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.5512 - val_loss: 0.6848
Epoch 9/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.5200 - val_loss: 0.6669
Epoch 10/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.4929 - val_loss: 0.6501
Epoch 11/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.4709 - val_loss: 0.6335
Epoch 12/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

In [ ]:

print(results.head(10))


# Predict a new house
new_house = pd.DataFrame([{
    "area": 7420,
    "bedrooms": 4,
    "bathrooms": 2,
    "stories": 3,
    "mainroad": 1,
    "guestroom": 0,
    "basement": 0,
    "hotwaterheating": 0,
    "airconditioning": 1,
    "parking": 2,
    "prefarea": 1,
    "furnishingstatus": 2
}])

new_house = new_house[X.columns]

new_house_scaled = X_scaler.transform(
    new_house
)

new_price_scaled = model.predict(
    new_house_scaled,
    verbose=0
)

new_price = y_scaler.inverse_transform(
    new_price_scaled
)[0][0]

print(f"Predicted House Price: {new_price:.2f}")

In [ ]:


# Save model and scalers
model.save("housing_ann_model.keras")

joblib.dump(
    X_scaler,
    "housing_X_scaler.pkl"
)

joblib.dump(
    y_scaler,
    "housing_y_scaler.pkl"
)

print("Model and scalers saved successfully")